# 比較デモ①：孤立 / 集約（動作確認・時間計測用）

連合学習に入る前の比較実験です。
- **孤立**：1クライアント分のデータだけで学習
- **集約**：全データを1箇所に集めて学習

両方とも**同じ共通テストセット**で評価し、モデルは新規初期化・シード固定で公平に比較します。
`run_simulation` を通さない素朴な実装なので、連合学習より速く終わります。

**目的**：数字（孤立 vs 集約の精度差）と実行時間を、手元で確認すること。


## 1. ライブラリのインストール（Colabのみ・毎回必要）

In [ ]:
!pip install -q "flwr-datasets[vision]>=0.5.0" torch torchvision matplotlib

## 2. モデル・データ処理の定義（task の中身）
データセットIDは正式名に修正済み。MNIST / Fashion-MNIST を切替できます。

In [ ]:
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Normalize, ToTensor

# dataset registry: short name -> (HF id, normalization)
DATASETS = {
    "mnist": ("ylecun/mnist", (0.1307,), (0.3081,)),
    "fashion_mnist": ("zalando-datasets/fashion_mnist", (0.2860,), (0.3530,)),
}

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32); self.pool1 = nn.MaxPool2d(2, 2); self.dropout1 = nn.Dropout(0.25)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64); self.pool2 = nn.MaxPool2d(2, 2); self.dropout2 = nn.Dropout(0.25)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128); self.dropout3 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x)))); x = self.dropout1(x)
        x = self.pool2(F.relu(self.bn2(self.conv2(x)))); x = self.dropout2(x)
        x = self.flatten(x); x = F.relu(self.fc1(x)); x = self.dropout3(x)
        return self.fc2(x)

fds = None
_current_dataset = None

def load_data(partition_id, num_partitions, dataset_name="mnist", random_seed=42):
    global fds, _current_dataset
    hf_id, mean, std = DATASETS[dataset_name]
    if fds is None or _current_dataset != dataset_name:
        partitioner = IidPartitioner(num_partitions=num_partitions)
        fds = FederatedDataset(dataset=hf_id, partitioners={"train": partitioner})
        _current_dataset = dataset_name
    partition = fds.load_partition(partition_id, "train")
    ptt = partition.train_test_split(test_size=0.2, seed=random_seed)
    tfm = Compose([ToTensor(), Normalize(mean, std)])
    def apply(b):
        b["image"] = [tfm(img) for img in b["image"]]; return b
    ptt = ptt.with_transform(apply)
    train_ds = ptt["train"]
    drop_last = len(train_ds) >= 32
    trainloader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=drop_last)
    testloader = DataLoader(ptt["test"], batch_size=32)
    return trainloader, testloader

def train(net, trainloader, epochs, device, learning_rate=0.001):
    net.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=learning_rate)
    crit = torch.nn.CrossEntropyLoss().to(device)
    net.train()
    losses = []
    for _ in range(epochs):
        rl, nb = 0.0, 0
        for batch in trainloader:
            images = batch["image"].to(device); labels = batch["label"].to(device)
            opt.zero_grad(); out = net(images); loss = crit(out, labels)
            loss.backward(); opt.step(); rl += loss.item(); nb += 1
        losses.append(rl / nb if nb else 0.0)
    return sum(losses) / len(losses) if losses else 0.0

def test(net, testloader, device):
    net.to(device)
    crit = torch.nn.CrossEntropyLoss()
    correct, loss_sum, total = 0, 0.0, 0
    net.eval()
    with torch.no_grad():
        for batch in testloader:
            images = batch["image"].to(device); labels = batch["label"].to(device)
            out = net(images)
            loss_sum += crit(out, labels).item() * len(labels); total += len(labels)
            correct += (torch.max(out.data, 1)[1] == labels).sum().item()
    return (loss_sum/total if total else 0.0), (correct/total if total else 0.0)

def get_weights(net):
    return [val.cpu().numpy() for _, val in net.state_dict().items()]

def set_weights(net, parameters):
    params_dict = zip(net.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    net.load_state_dict(state_dict, strict=True)


## 3. 比較実験の関数（compare の中身）
`run_isolated`（孤立）と `run_centralized`（集約）を定義します。

In [ ]:
"""比較デモ①: 孤立 / 集約 の学習・評価（run_simulationを通さない素朴な実装）"""
import torch
from torch.utils.data import DataLoader, ConcatDataset
from torchvision.transforms import Compose, Normalize, ToTensor
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner
# （Net, train, test, DATASETS は上のセルで定義済み）

def _build(dataset_name, num_partitions, seed):
    """全パーティションをまとめて取得し、共通テストセットと各クライアントの訓練データを作る"""
    hf_id, mean, std = DATASETS[dataset_name]
    partitioner = IidPartitioner(num_partitions=num_partitions)
    fds = FederatedDataset(dataset=hf_id, partitioners={"train": partitioner})
    tfm = Compose([ToTensor(), Normalize(mean, std)])
    def apply(b):
        b["image"] = [tfm(img) for img in b["image"]]; return b

    per_client_train = []
    test_parts = []
    for pid in range(num_partitions):
        part = fds.load_partition(pid, "train")
        ptt = part.train_test_split(test_size=0.2, seed=seed).with_transform(apply)
        per_client_train.append(ptt["train"])
        test_parts.append(ptt["test"])
    # 共通テストセット = 全クライアントのtestを連結（3実験で共通利用）
    common_test = ConcatDataset(test_parts)
    return per_client_train, common_test

def run_isolated(dataset_name="mnist", num_partitions=5, epochs=3,
                 learning_rate=0.001, seed=42, client_id=0):
    """孤立: 1クライアント分だけで学習"""
    torch.manual_seed(seed)
    device = torch.device("cpu")
    per_client_train, common_test = _build(dataset_name, num_partitions, seed)
    train_ds = per_client_train[client_id]
    trainloader = DataLoader(train_ds, batch_size=32, shuffle=True,
                             drop_last=len(train_ds) >= 32)
    testloader = DataLoader(common_test, batch_size=32)
    net = Net()
    train(net, trainloader, epochs, device, learning_rate=learning_rate)
    loss, acc = test(net, testloader, device)
    return {"setting": "isolated", "train_size": len(train_ds), "accuracy": acc, "loss": loss}

def run_centralized(dataset_name="mnist", num_partitions=5, epochs=3,
                    learning_rate=0.001, seed=42):
    """集約: 全クライアント分をまとめて学習"""
    torch.manual_seed(seed)
    device = torch.device("cpu")
    per_client_train, common_test = _build(dataset_name, num_partitions, seed)
    all_train = ConcatDataset(per_client_train)
    trainloader = DataLoader(all_train, batch_size=32, shuffle=True, drop_last=True)
    testloader = DataLoader(common_test, batch_size=32)
    net = Net()
    train(net, trainloader, epochs, device, learning_rate=learning_rate)
    loss, acc = test(net, testloader, device)
    return {"setting": "centralized", "train_size": len(all_train), "accuracy": acc, "loss": loss}


## 4. 実行①：孤立（1クライアント分だけで学習）
`num_partitions=5` なら全体の1/5のデータで学習します。`%%time` で時間が出ます。

In [ ]:
%%time
r_iso = run_isolated(dataset_name="mnist", num_partitions=5, epochs=1,
                     learning_rate=0.001, seed=42, client_id=0)
print(r_iso)

## 5. 実行②：集約（全データで学習）
**注意**：データ量が多いぶん時間がかかります（CPUだと1エポックで1分以上のことも）。

In [ ]:
%%time
r_cen = run_centralized(dataset_name="mnist", num_partitions=5, epochs=1,
                        learning_rate=0.001, seed=42)
print(r_cen)

## 6. 結果を並べて表示

In [ ]:
print(f"{'設定':12s} {'データ量':>8s} {'精度':>8s}")
for r in [r_iso, r_cen]:
    print(f"{r['setting']:12s} {r['train_size']:8d} {r['accuracy']:8.4f}")

## 7. いろいろ試す（任意）
孤立と集約の精度差は、`num_partitions`（クライアント数）を増やすと広がります。
例えば 20 にすると、孤立は全体の1/20のデータになり、精度がはっきり落ちます。
データセットを `"fashion_mnist"` にすると難易度が上がり、差が出やすくなります。

In [ ]:
%%time
# 例：20分割にして孤立を不利にしてみる
r_iso20 = run_isolated(dataset_name="mnist", num_partitions=20, epochs=1, seed=42)
print(r_iso20)